In [1]:
import sys
from lightfm import LightFM

print("Python:", sys.version)
print("LightFM installed successfully!")

Python: 3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
LightFM installed successfully!


C:\Users\aruna\anaconda3\envs\lightfm_env\Lib\site-packages\lightfm\_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [2]:
import os
import random
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import precision_at_k, recall_at_k, auc_score
from lightfm.cross_validation import random_train_test_split

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
BASE_DIR = os.getcwd()

DATASET_DIR = os.path.join(
    BASE_DIR,
    "Dataset",
    "ml-100k"
)

RATINGS_PATH = os.path.join(DATASET_DIR, "u.data")
ITEMS_PATH = os.path.join(DATASET_DIR, "u.item")
USERS_PATH = os.path.join(DATASET_DIR, "u.user")

print("Dataset directory:", DATASET_DIR)
print("Ratings file exists:", os.path.exists(RATINGS_PATH))
print("Items file exists:", os.path.exists(ITEMS_PATH))
print("Users file exists:", os.path.exists(USERS_PATH))

Dataset directory: C:\Users\aruna\LightFM_Recommendation_System\Dataset\ml-100k
Ratings file exists: True
Items file exists: True
Users file exists: True


In [4]:
ratings_columns = [
    "user_id",
    "movie_id",
    "rating",
    "timestamp"
]

ratings_df = pd.read_csv(
    RATINGS_PATH,
    sep="\t",
    names=ratings_columns,
    encoding="latin-1"
)

print("Ratings shape:", ratings_df.shape)
ratings_df.head()

Ratings shape: (100000, 4)


,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [5]:
movie_columns = [
    "movie_id",
    "title",
    "release_date",
    "video_release_date",
    "imdb_url",
    "unknown",
    "Action",
    "Adventure",
    "Animation",
    "Children",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Fantasy",
    "Film-Noir",
    "Horror",
    "Musical",
    "Mystery",
    "Romance",
    "Sci-Fi",
    "Thriller",
    "War",
    "Western"
]

movies_df = pd.read_csv(
    ITEMS_PATH,
    sep="|",
    names=movie_columns,
    encoding="latin-1"
)

print("Movies shape:", movies_df.shape)
movies_df.head()

Movies shape: (1682, 24)


,movie_id,title,release_date,video_release_date,imdb_url,unknown,Action,Adventure,Animation,Children,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [6]:
user_columns = [
    "user_id",
    "age",
    "gender",
    "occupation",
    "zip_code"
]

users_df = pd.read_csv(
    USERS_PATH,
    sep="|",
    names=user_columns,
    encoding="latin-1"
)

print("Users shape:", users_df.shape)
users_df.head()

Users shape: (943, 5)


,user_id,age,gender,occupation,zip_code
0,1,24,M,technician,85711
1,2,53,F,other,94043
2,3,23,M,writer,32067
3,4,24,M,technician,43537
4,5,33,F,other,15213


In [7]:
print("Missing values in ratings:")
print(ratings_df.isnull().sum())

print("\nMissing values in movies:")
print(movies_df.isnull().sum().head())

print("\nMissing values in users:")
print(users_df.isnull().sum())

print("\nDuplicate ratings:", ratings_df.duplicated().sum())
print("Duplicate movies:", movies_df.duplicated().sum())
print("Duplicate users:", users_df.duplicated().sum())

Missing values in ratings:
user_id      0
movie_id     0
rating       0
timestamp    0
dtype: int64

Missing values in movies:
movie_id                 0
title                    0
release_date             1
video_release_date    1682
imdb_url                 3
dtype: int64

Missing values in users:
user_id       0
age           0
gender        0
occupation    0
zip_code      0
dtype: int64

Duplicate ratings: 0
Duplicate movies: 0
Duplicate users: 0


In [8]:
# Remove duplicate records, if any
ratings_df = ratings_df.drop_duplicates().copy()
movies_df = movies_df.drop_duplicates(subset=["movie_id"]).copy()
users_df = users_df.drop_duplicates(subset=["user_id"]).copy()

# Fill missing movie titles safely
movies_df["title"] = movies_df["title"].fillna("Unknown Movie")

# Convert timestamp into readable datetime
ratings_df["rating_datetime"] = pd.to_datetime(
    ratings_df["timestamp"],
    unit="s"
)

print("Cleaned ratings shape:", ratings_df.shape)
print("Cleaned movies shape:", movies_df.shape)
print("Cleaned users shape:", users_df.shape)

Cleaned ratings shape: (100000, 5)
Cleaned movies shape: (1682, 24)
Cleaned users shape: (943, 5)


In [9]:
print("Number of users:", ratings_df["user_id"].nunique())
print("Number of movies:", ratings_df["movie_id"].nunique())
print("Number of ratings:", len(ratings_df))
print("Average rating:", round(ratings_df["rating"].mean(), 2))
print("Minimum rating:", ratings_df["rating"].min())
print("Maximum rating:", ratings_df["rating"].max())

Number of users: 943
Number of movies: 1682
Number of ratings: 100000
Average rating: 3.53
Minimum rating: 1
Maximum rating: 5


In [ ]:
rating_distribution = (
    ratings_df["rating"]
    .value_counts()
    .sort_index()
)

print(rating_distribution)

plt.figure(figsize=(8, 5))
rating_distribution.plot(kind="bar")

plt.title("Movie Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Ratings")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

rating
1     6110
2    11370
3    27145
4    34174
5    21201
Name: count, dtype: int64


In [ ]:
movie_statistics = (
    ratings_df.groupby("movie_id")
    .agg(
        rating_count=("rating", "count"),
        average_rating=("rating", "mean")
    )
    .reset_index()
)

movie_statistics = movie_statistics.merge(
    movies_df[["movie_id", "title"]],
    on="movie_id",
    how="left"
)

most_rated_movies = movie_statistics.sort_values(
    "rating_count",
    ascending=False
).head(10)

most_rated_movies[
    ["title", "rating_count", "average_rating"]
]

In [ ]:
positive_ratings_df = ratings_df[
    ratings_df["rating"] >= 4
].copy()

print("All ratings:", len(ratings_df))
print("Positive interactions:", len(positive_ratings_df))
print(
    "Positive interaction percentage:",
    round(
        len(positive_ratings_df) / len(ratings_df) * 100,
        2
    ),
    "%"
)

In [ ]:
lightfm_dataset = Dataset()

lightfm_dataset.fit(
    users=users_df["user_id"].unique(),
    items=movies_df["movie_id"].unique()
)

num_users, num_items = lightfm_dataset.interactions_shape()

print("Number of mapped users:", num_users)
print("Number of mapped movies:", num_items)